## Computational Carpentry Project

**Group G — Team members**

- Marie Lacroix
- Chloé Baruselli
- Bertille Delloye
- Enéa Drezet--Marçot


In [ ]:
import pandas as pd
import re
import numpy as np
from fractions import Fraction
import math
import matplotlib.pyplot as plt

### Part A - Data structures and functions 

#### 1. Load periodic table

In [ ]:
df = pd.read_csv("periodic_table.csv")

#### 2. Create python dictionary

In [ ]:
symbol_to_mass = dict(zip(df["Symbol"], df["AtomicMass"]))

#### 3. Function for molecular masses

I use a regular expression `"([A-Z][a-z]*)(\d*)"` to extract each symbol/count pair from the formula string:
- `"[A-Z][a-z]*"` matches an element symbol: one uppercase letter followed by zero or more lowercase letters (so `"Ca"` is read as one symbol, not `"C" + "a"`).
- `"\d*"` matches the digits that follow, if any.

For each pair found, I look up the atomic mass in `symbol_to_mass` and multiply it by the atom count, summing everything into `total_mass`.

In [ ]:
def molecular_mass(formula, mass_dict=symbol_to_mass):
    pattern = r"([A-Z][a-z]*)(\d*)"
    
    total_mass = 0.0
    for symbol, count in re.findall(pattern, formula):
        if symbol == "":
            continue
        count = int(count) if count else 1
        if symbol not in mass_dict:
            raise ValueError(f"Unknown element symbol: {symbol}")
        total_mass += mass_dict[symbol] * count
    
    return total_mass

In [ ]:
print(molecular_mass("H2O"))
print(molecular_mass("C6H12O6"))

This approach is limited because it treats the formula as a flat list of symbol/count pairs; there is no grouping. This becomes a problem for formulas containing parentheses.

#### 4. Extend the function

I rewrote the parser to walk through the formula character by character, using recursion to handle parentheses:
- When the parser encounters "(", it calls itself on the substring starting right after the "(". This recursive call keeps consuming characters until it hits the matching ")".
- When it hits ")", it stops and returns the accumulated mass of everything inside that group, along with the index just past the ")".
- Back in the caller, the code then checks for a number right after the ")" and multiplies the group's mass by it.
- Element symbols outside parentheses are parsed the same way as before, just using helper functions ("_read_symbol", "_read_number").

Recursion is a natural fit here because parentheses can be nested (e.g. "K4(Fe(CN)6)"). It lets the same function handle any depth of nesting without extra code.

In [ ]:
def molecular_mass_recursive(formula, mass_dict = symbol_to_mass):
    total_mass, _ = _parse_formula(formula, 0, mass_dict)
    return total_mass


def _parse_formula(formula, index, mass_dict):
    total_mass = 0.0

    while index < len(formula):
        character = formula[index]

        if character == "(":
            inner_mass, index = _parse_formula(formula, index + 1, mass_dict)
            count, index = _read_number(formula, index)
            total_mass += inner_mass * count

        elif character == ")":
            return total_mass, index + 1

        else:
            symbol, index = _read_symbol(formula, index)
            count, index = _read_number(formula, index)
            if symbol not in mass_dict:
                raise ValueError(f"Unknown element symbol: {symbol}")
            total_mass += mass_dict[symbol] * count

    return total_mass, index


def _read_symbol(formula, index):
    match = re.match(r"[A-Z][a-z]*", formula[index:])
    symbol = match.group()
    return symbol, index + len(symbol)


def _read_number(formula, index):
    match = re.match(r"\d*", formula[index:])
    number_str = match.group()
    count = int(number_str) if number_str else 1
    return count, index + len(number_str)

In [ ]:
print(molecular_mass_recursive("Ca(OH)2"))
print(molecular_mass_recursive("Mg(NO3)2"))

#### 5. Extend the function
I added a pre-processing step:
1. Split the formula on `"."`, producing one or more independent chunks (e.g. `"CuSO4.5H2O"` into `["CuSO4", "5H2O"]`).
2. For each chunk, check if it starts with digits using `"^(\d+)(.*)"`. If so, those digits are the coefficient, and the rest of the chunk is the formula to parse (e.g. `"5H2O"` → coefficient 5, formula `"H2O"`). If there is no leading number, the coefficient defaults to 1.
3. Each chunk's formula is parsed independently using the same recursive parser from before — no changes were needed there.
4. Each chunk's mass is multiplied by its coefficient, and all chunks are summed into the total molecular mass.

In [ ]:
def molecular_mass_recursive_2(formula, mass_dict=symbol_to_mass):
    total_mass = 0.0
    
    parts = formula.split(".")
    
    for part in parts:
        match = re.match(r"^(\d+)(.*)", part)
        if match:
            coefficient = int(match.group(1))
            rest_of_formula = match.group(2)
        else:
            coefficient = 1
            rest_of_formula = part
        
        part_mass, _ = _parse_formula(rest_of_formula, 0, mass_dict)
        total_mass += coefficient * part_mass
    
    return total_mass

In [ ]:
print(molecular_mass_recursive_2("CuSO4.5H2O"))
print(molecular_mass_recursive_2("Na2CO3.10H2O"))

### Part B - Stoichiometry and reaction balancing
#### 1. Reaction balancer function

Each chemical element gives one equation: the total atoms of that element on the reactant side must equal the total atoms on the product side. If I treat product counts as negative reactant counts, this becomes: for every element, the weighted sum of atom counts must equal zero.

This is naturally expressed as a matrix A where:
- each row is a chemical element,
- each column is a species (a reactant or a product),
- each entry is how many atoms of that element are in that species (positive for reactants, negative for products).

Finding the coefficients then means finding a vector x (the coefficients) such that $Ax = 0$.

In [ ]:
def _parse_formula_simple(formula):
    pattern = r"([A-Z][a-z]*)(\d*)"
    counts = {}
    for symbol, count in re.findall(pattern, formula):
        if symbol == "":
            continue
        count = int(count) if count else 1
        counts[symbol] = counts.get(symbol, 0) + count
    return counts


def balance_reaction(reactants, products):
    species = reactants + products
    parsed = [_parse_formula_simple(f) for f in species]
    elements = sorted(set(el for d in parsed for el in d))

    A = np.zeros((len(elements), len(species)))
    for j, d in enumerate(parsed):
        sign = 1 if j < len(reactants) else -1
        for el, cnt in d.items():
            i = elements.index(el)
            A[i, j] += sign * cnt
    
    u, s, vh = np.linalg.svd(A)
    tol = 1e-10
    rank = np.sum(s > tol)
    null_space_vector = vh[rank:][0]
    
    smallest = np.min(np.abs(null_space_vector[np.abs(null_space_vector) > tol]))
    normalized = null_space_vector / smallest
    
    fractions = [Fraction(c).limit_denominator(10) for c in normalized]
    denominators = [f.denominator for f in fractions]
    scale = np.lcm.reduce(denominators)
    
    coeffs = [int(round(c * scale)) for c in normalized]
    
    if coeffs[0] < 0:
        coeffs = [-c for c in coeffs]
    
    return coeffs

In [ ]:
# 1. Water formation: expect [2, 1, 2] -> 2H2 + O2 -> 2H2O
print(balance_reaction(["H2", "O2"], ["H2O"]))

# 2. Methane combustion: expect [1, 2, 1, 2] -> CH4 + 2O2 -> CO2 + 2H2O
print(balance_reaction(["CH4", "O2"], ["CO2", "H2O"]))

# 3. Iron rusting: expect [4, 3, 2] -> 4Fe + 3O2 -> 2Fe2O3
print(balance_reaction(["Fe", "O2"], ["Fe2O3"]))

#### 2. Mass conservation check
The function `check_mass_conservation` takes two dictionaries: one for reactants and one for products, where each key is a chemical formula (e.g. `"H2"`) and each value is its stoichiometric coefficient (e.g. 2). For each side of the reaction:
1. Compute the molecular mass of each formula using `molecular_mass` from Part A.
2. Multiply that mass by its coefficient.
3. Sum these values to get the total mass on that side of the reaction.

The two totals (reactant side and product side) are then compared. Because the masses involved are floating-point numbers, I do not check for exact equality, since floating-point arithmetic can introduce tiny rounding errors even when two quantities are mathematically equal. Instead, I use `math.isclose` with a small tolerance, which considers two numbers equal enough if they are within a very small relative difference of each other.

The function returns `True` if the masses match within tolerance, `False` otherwise, and also prints a human-readable message stating whether the reaction is balanced.

In [ ]:
def check_mass_conservation(reactants_dict, products_dict, mass_dict=symbol_to_mass):

    reactant_mass = sum(
        molecular_mass(formula, mass_dict) * coeff
        for formula, coeff in reactants_dict.items()
    )
    product_mass = sum(
        molecular_mass(formula, mass_dict) * coeff
        for formula, coeff in products_dict.items()
    )

    is_balanced = math.isclose(reactant_mass, product_mass, rel_tol=1e-6)

    if is_balanced:
        print(f"Reaction is balanced: {reactant_mass:.4f} g/mol on both sides.")
    else:
        print(
            f"Reaction is NOT balanced: reactants = {reactant_mass:.4f} g/mol, "
            f"products = {product_mass:.4f} g/mol."
        )

    return is_balanced

In [ ]:
# Verify mass conservation for the three reactions balanced above
check_mass_conservation({"H2": 2, "O2": 1}, {"H2O": 2})
check_mass_conservation({"CH4": 1, "O2": 2}, {"CO2": 1, "H2O": 2})
check_mass_conservation({"Fe": 4, "O2": 3}, {"Fe2O3": 2})

This step reinforced that "balancing by atoms" and "balancing by mass" are really the same underlying fact, just viewed from two different angles — mass conservation is a consequence of atom conservation, since every atom carries a fixed mass regardless of what molecule it is part of.

### Part C – Simulation / Modeling

We are going to answer two questions by using the Monte Carlo method.


#### 1. Monte Carlo estimation of $\pi$

We consider a square of side 2, with x and y coordinates between -1 and 1.

Inside this square, there is a unit circle centered at the origin.

The probability that a random point falls inside the circle is:

$$
P(\text{inside circle}) = \frac{A_{circle}}{A_{square}} = \frac{\pi}{4}
$$

If we generate N random points and count how many are inside the circle, we can estimate $\pi$ using:

$$
\pi \approx 4 \times \frac{N_{inside}}{N}
$$

In [ ]:
def estimate_pi(N):
    x = np.random.uniform(-1, 1, N)
    y = np.random.uniform(-1, 1, N)

    inside_circle = x**2 + y**2 <= 1
    number_inside = np.sum(inside_circle)
    pi_estimate = 4 * number_inside / N

    return pi_estimate

In this function, we first generate N random x and y coordinates between -1 and 1. We know that a point is inside the unit circle if $x^2+y^2\leq1$. We then count the number of points inside the circle and estimate $\pi$.

We can then test our function to see if we get a value close to the real one and calculate the absolute error.

In [ ]:
N = 10000

pi_estimate = estimate_pi(N)

print("Number of points:", N)
print("Estimated value of pi:", pi_estimate)
print("Actual value of pi:", np.pi)
print("Absolute error:", abs(pi_estimate - np.pi))

N = 30000

pi_estimate = estimate_pi(N)

print("Number of points:", N)
print("Estimated value of pi:", pi_estimate)
print("Actual value of pi:", np.pi)
print("Absolute error:", abs(pi_estimate - np.pi))

The result is close to the actual value of $\pi$, but it is not exactly equal to it because the method is based on random sampling.

In general, increasing the number of random points should improve the estimate because the proportion of points inside the circle becomes more representative of the theoretical probability. To visualize this accuracy improvement, we can plot the convergence of our estimate versus $N$.

In [ ]:
N_values = [
    10,
    50,
    100,
    500,
    1000,
    5000,
    10000,
    50000,
    100000
]
pi_estimates = []

for N in N_values:
    estimate = estimate_pi(N)
    pi_estimates.append(estimate)

plt.figure(figsize=(8, 5))

plt.plot(N_values, pi_estimates, marker="o", label="Monte Carlo estimate")
plt.axhline(np.pi, linestyle="--", label="Actual π")

plt.xscale("log")

plt.xlabel("Number of random points N")
plt.ylabel("Estimated value of π")

plt.legend()
plt.grid()

plt.show()

We used a logarithmic x-axis because the values range from 10 to 100000.

For small values of N, the estimate of $\pi$ can be quite different from the true value because only a small number of random points are used.

As N increases, the estimate generally becomes closer to π = 3.14159.

The convergence is not perfectly smooth because every simulation contains randomness. Even with a larger N, the estimated value can sometimes move slightly farther away from π before becoming closer again.



#### 2. Chemistry-inspired Monte Carlo

In this simplified model, we simulate M molecular collisions. A random energy is assigned to each collision.

We then define a threshold energy. If the collision energy is greater than or equal to this threshold, we consider the collision to be energetic enough to trigger a reaction.

The fraction

$$
\frac{\text{successful collisions}}{\text{total collisions}}
$$

is therefore interpreted as an estimate of the reaction probability.

In [ ]:
def collision_simulation(M, threshold):
    energies = np.random.uniform(0, 100, M)

    successful_collisions = energies >= threshold

    number_successful = np.sum(successful_collisions)

    reaction_probability = number_successful / M

    return reaction_probability, energies

In this function, we generate M random collision energies between $0$ and $100$. We identify the collisions with enough energy to react, count the successful collisions, and then estimate the reaction probability. The function is tested in the following cell.

In [ ]:
M = 10000
threshold_energy = 60

reaction_probability, energies = collision_simulation(M, threshold_energy)

print("Number of collisions:", M)
print("Threshold energy:", threshold_energy)
print("Number of collisions above threshold:", np.sum(energies >= threshold_energy))
print("Estimated reaction probability:", reaction_probability)

For this simulation, collision energies were chosen uniformly between 0 and 100. The threshold energy was set to 60. Therefore, only collisions with an energy greater than or equal to 60 were considered successful. Since the energies are uniformly distributed, we expect approximately 40% of the collisions to have energies between 60 and 100. The Monte Carlo result should therefore be close to 0.40 when M is large. This is the case.